# EDA > Pivot

<div class="alert alert-info">Create pivot tables, frequency tables, and crosstabs</div>

The `pivot` function creates frequency tables for a single variable or crosstabs for two variables. It supports various aggregation functions, normalization, and totals.

<!-- pyrsm-teaching-note: roadmap -->
## Teaching Roadmap

Use this notebook to teach how a pivot table changes the unit of analysis. Students should identify the row variable, column variable, values variable, and aggregation function. A useful check is whether each displayed number is a count, total, mean, or percentage.


In [1]:
import polars as pl
import pyrsm as rsm

## setup pyrsm for autoreload
%reload_ext autoreload
%autoreload 2
%aimport pyrsm

# Diamonds Dataset

In [2]:
diamonds = pl.read_parquet("https://github.com/radiant-ai-hub/pyrsm/raw/refs/heads/main/examples/data/data/diamonds.parquet")
diamonds

price,carat,clarity,cut,color,depth,table,x,y,z,date
i32,f64,enum,enum,enum,f64,f64,f64,f64,f64,date
580,0.32,"""VS1""","""Ideal""","""H""",61.0,56.0,4.43,4.45,2.71,2012-02-26
650,0.34,"""SI1""","""Very Good""","""G""",63.4,57.0,4.45,4.42,2.81,2012-02-26
630,0.3,"""VS2""","""Very Good""","""G""",63.1,58.0,4.27,4.23,2.68,2012-02-26
706,0.35,"""VVS2""","""Ideal""","""H""",59.2,56.0,4.6,4.65,2.74,2012-02-26
1080,0.4,"""VS2""","""Premium""","""F""",62.6,58.0,4.72,4.68,2.94,2012-02-26
…,…,…,…,…,…,…,…,…,…,…
4173,1.14,"""SI1""","""Very Good""","""J""",63.3,55.0,6.6,6.67,4.2,2015-12-01
8396,1.51,"""SI1""","""Ideal""","""I""",61.2,60.0,7.39,7.37,4.52,2015-12-01
449,0.32,"""VS2""","""Premium""","""I""",62.6,58.0,4.37,4.42,2.75,2015-12-01


In [3]:
rsm.md("https://raw.githubusercontent.com/radiant-ai-hub/pyrsm/refs/heads/main/examples/data/data/diamonds_description.md")

Error fetching URL: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>

## Frequency Table (Single Variable)

Count occurrences of each value in a categorical column.

In [4]:
rsm.eda.pivot(diamonds, rows="cut")

cut,count
enum,u32
"""Ideal""",1176
"""Good""",275
"""Very Good""",677
"""Fair""",101
"""Premium""",771


In [5]:
rsm.eda.pivot(diamonds, rows="color")

color,count
enum,u32
"""H""",454
"""J""",164
"""G""",597
"""I""",284
"""F""",565
"""E""",554
"""D""",382


In [6]:
rsm.eda.pivot(diamonds, rows="cut", values="price")

cut,price_mean
enum,f64
"""Good""",4130.432727
"""Very Good""",3959.915805
"""Premium""",4369.40856
"""Fair""",4505.237624
"""Ideal""",3470.223639


## Frequency Table with Percentages

In [7]:
rsm.eda.pivot(diamonds, rows="cut", normalize="total")

cut,count,count_pct
enum,u32,f64
"""Premium""",771,25.7
"""Ideal""",1176,39.2
"""Very Good""",677,22.566667
"""Fair""",101,3.366667
"""Good""",275,9.166667


## Frequency Table with Totals

In [8]:
rsm.eda.pivot(diamonds, rows="cut", totals=True)

cut,count
str,f64
"""Good""",275.0
"""Premium""",771.0
"""Ideal""",1176.0
"""Fair""",101.0
"""Very Good""",677.0
"""Total""",3000.0


## Crosstab (Two Variables)

Cross-tabulate two categorical variables to see their joint distribution.

In [9]:
rsm.eda.pivot(diamonds, rows="cut", cols="color")

cut,I,E,D,G,H,F,J
enum,f64,f64,f64,f64,f64,f64,f64
"""Ideal""",105.0,194.0,163.0,254.0,169.0,238.0,53.0
"""Premium""",83.0,144.0,92.0,156.0,132.0,119.0,45.0
"""Good""",26.0,62.0,35.0,40.0,37.0,55.0,20.0
"""Fair""",11.0,14.0,15.0,16.0,21.0,17.0,7.0
"""Very Good""",59.0,140.0,77.0,131.0,95.0,136.0,39.0


## Crosstab with Totals

In [10]:
rsm.eda.pivot(diamonds, rows="cut", cols="color", totals=True)

cut,D,F,G,H,I,J,E,Total
str,f64,f64,f64,f64,f64,f64,f64,f64
"""Premium""",92.0,119.0,156.0,132.0,83.0,45.0,144.0,771.0
"""Fair""",15.0,17.0,16.0,21.0,11.0,7.0,14.0,101.0
"""Ideal""",163.0,238.0,254.0,169.0,105.0,53.0,194.0,1176.0
"""Good""",35.0,55.0,40.0,37.0,26.0,20.0,62.0,275.0
"""Very Good""",77.0,136.0,131.0,95.0,59.0,39.0,140.0,677.0
"""Total""",382.0,565.0,597.0,454.0,284.0,164.0,554.0,3000.0


## Row Normalization

Show percentages within each row (rows sum to 100%).

In [11]:
rsm.eda.pivot(diamonds, rows="cut", cols="color", normalize="row", totals=True)

cut,J,E,I,G,F,H,D,Total
str,f64,f64,f64,f64,f64,f64,f64,f64
"""Premium""",5.836576,18.677043,10.76524,20.233463,15.434501,17.120623,11.932555,100.0
"""Very Good""",5.760709,20.679468,8.714919,19.350074,20.088626,14.032496,11.373708,100.0
"""Fair""",6.930693,13.861386,10.891089,15.841584,16.831683,20.792079,14.851485,100.0
"""Good""",7.272727,22.545455,9.454545,14.545455,20.0,13.454545,12.727273,100.0
"""Ideal""",4.506803,16.496599,8.928571,21.598639,20.238095,14.370748,13.860544,100.0
"""Total""",5.466667,18.466667,9.466667,19.9,18.833333,15.133333,12.733333,100.0


## Column Normalization

Show percentages within each column (columns sum to 100%).

In [12]:
rsm.eda.pivot(diamonds, rows="cut", cols="color", normalize="column")

cut,E,F,H,D,J,I,G
enum,f64,f64,f64,f64,f64,f64,f64
"""Good""",11.191336,9.734513,8.14978,9.162304,12.195122,9.15493,6.700168
"""Ideal""",35.018051,42.123894,37.22467,42.670157,32.317073,36.971831,42.546064
"""Fair""",2.527076,3.00885,4.625551,3.926702,4.268293,3.873239,2.680067
"""Very Good""",25.270758,24.070796,20.92511,20.157068,23.780488,20.774648,21.943049
"""Premium""",25.99278,21.061947,29.07489,24.08377,27.439024,29.225352,26.130653


## Aggregation with Values

Instead of counting, aggregate a numeric variable by groups.

In [13]:
# Mean price by cut and color
rsm.eda.pivot(diamonds, rows="cut", cols="color", values="price", agg="mean")

cut,D,H,J,I,G,E,F
enum,f64,f64,f64,f64,f64,f64,f64
"""Good""",3436.514286,3958.162162,3837.25,6147.346154,5116.225,3847.209677,3443.4
"""Ideal""",2667.478528,3515.674556,4987.754717,4330.352381,3844.535433,2851.659794,3375.054622
"""Fair""",4582.733333,5742.47619,6102.0,2676.727273,3919.8125,3149.928571,5101.294118
"""Very Good""",3299.974026,4207.294737,5212.410256,5409.881356,3864.274809,3566.442857,3669.727941
"""Premium""",3814.98913,5066.295455,7515.466667,5056.686747,3976.5,3364.694444,4086.831933


In [14]:
# Median carat by cut and color
rsm.eda.pivot(diamonds, rows="cut", cols="color", values="carat", agg="median")

cut,D,H,J,E,F,G,I
enum,f64,f64,f64,f64,f64,f64,f64
"""Premium""",0.69,1.02,1.51,0.52,0.71,0.71,1.01
"""Ideal""",0.51,0.7,1.07,0.51,0.54,0.535,0.7
"""Good""",0.7,1.0,0.865,0.7,0.7,1.0,1.36
"""Very Good""",0.54,0.9,1.04,0.71,0.7,0.72,1.01
"""Fair""",0.9,1.01,1.0,0.715,1.0,0.855,0.73


# Titanic Dataset

In [15]:
titanic = pl.read_parquet("https://github.com/radiant-ai-hub/pyrsm/raw/refs/heads/main/examples/data/data/titanic.parquet")
titanic.head()

pclass,survived,sex,age,sibsp,parch,fare,name,cabin,embarked
enum,enum,enum,f64,i32,i32,f64,str,str,enum
"""1st""","""Yes""","""female""",29.0,0,0,211.337494,"""Allen, Miss. Elisabeth Walton""","""B5""","""Southampton"""
"""1st""","""Yes""","""male""",0.9167,1,2,151.550003,"""Allison, Master. Hudson Trevor""","""C22 C26""","""Southampton"""
"""1st""","""No""","""female""",2.0,1,2,151.550003,"""Allison, Miss. Helen Loraine""","""C22 C26""","""Southampton"""
"""1st""","""No""","""male""",30.0,1,2,151.550003,"""Allison, Mr. Hudson Joshua Cre…","""C22 C26""","""Southampton"""
"""1st""","""No""","""female""",25.0,1,2,151.550003,"""Allison, Mrs. Hudson J C (Bess…","""C22 C26""","""Southampton"""


In [16]:
rsm.md("https://raw.githubusercontent.com/radiant-ai-hub/pyrsm/refs/heads/main/examples/data/data/titanic_description.md")

Error fetching URL: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>

## Survival by Passenger Class

In [17]:
rsm.eda.pivot(titanic, rows="pclass", cols="survived", totals=True)

pclass,Yes,No,Total
str,f64,f64,f64
"""2nd""",115.0,146.0,261.0
"""1st""",179.0,103.0,282.0
"""3rd""",131.0,369.0,500.0
"""Total""",425.0,618.0,1043.0


## Survival Rate by Class (Row Percentages)

In [18]:
rsm.eda.pivot(titanic, rows="pclass", cols="survived", normalize="row", totals=True)

pclass,No,Yes,Total
str,f64,f64,f64
"""3rd""",73.8,26.2,100.0
"""1st""",36.524823,63.475177,100.0
"""2nd""",55.938697,44.061303,100.0
"""Total""",59.252157,40.747843,100.0


## Survival by Sex

In [19]:
rsm.eda.pivot(titanic, rows="sex", cols="survived", normalize="row", totals=True)

sex,No,Yes,Total
str,f64,f64,f64
"""female""",24.870466,75.129534,100.0
"""male""",79.452055,20.547945,100.0
"""Total""",59.252157,40.747843,100.0


## Embarkation Port Distribution

In [20]:
rsm.eda.pivot(titanic, rows="embarked", normalize="total", totals=True)

embarked,count,count_pct
str,f64,f64
"""Cherbourg""",212.0,20.325983
"""Southampton""",781.0,74.880153
"""Queenstown""",50.0,4.793864
"""Total""",1043.0,100.0


## Mean Fare by Class and Survival

In [21]:
rsm.eda.pivot(titanic, rows="pclass", cols="survived", values="fare", agg="mean")

pclass,Yes,No
enum,f64,f64
"""1st""",102.465226,74.678276
"""2nd""",23.180471,20.811044
"""3rd""",12.427449,13.039712


© Vincent Nijs (2026)

## Additional examples


In [22]:
# Normalized crosstab with totals
rsm.eda.pivot(diamonds, rows="cut", cols="color", normalize="row", totals=True)


cut,J,I,F,H,D,E,G,Total
str,f64,f64,f64,f64,f64,f64,f64,f64
"""Premium""",5.836576,10.76524,15.434501,17.120623,11.932555,18.677043,20.233463,100.0
"""Fair""",6.930693,10.891089,16.831683,20.792079,14.851485,13.861386,15.841584,100.0
"""Good""",7.272727,9.454545,20.0,13.454545,12.727273,22.545455,14.545455,100.0
"""Ideal""",4.506803,8.928571,20.238095,14.370748,13.860544,16.496599,21.598639,100.0
"""Very Good""",5.760709,8.714919,20.088626,14.032496,11.373708,20.679468,19.350074,100.0
"""Total""",5.466667,9.466667,18.833333,15.133333,12.733333,18.466667,19.9,100.0
